# Detector evaluation and cleanup: EX (ex_run1) and HE (he_run5)

**Task.** Stage 1: measure the production lesion detectors properly (locked test sets, healthy eyes, error analysis). Stage 2: test fixes that need no retraining, choose by a pre-specified rule on validation data, confirm once on the test sets. Ownership: Victoria (decision.md 2026-11-02, "we own the detector cleanup").

**How to use.** Run All. Settings live in the Parameters cell only. Locally the notebook builds the fixed manifest and (optionally) a package for RunPod; on the pod it runs the inference (GPU), which is cached, so every later run only analyses. Outputs are overwritten in `notebooks/outputs/detector_evaluation/`.

**Rules.** No hand-typed results in markdown; every reported figure comes from the generated Claims table. Hard checks stop the notebook: the record's numbers must reproduce before anything new counts; three independent matching paths must agree; checkpoints are identified by size, epoch and recall.

## Pre-specified questions, data and decision rules

**Production configuration (the baseline).** As used for the holdout detector controls (`scratch/build_test_holdout.py`): CLAHE (LAB L), full-resolution sliding window (EX 512/384, HE 768/640), threshold 0.35, EX with learned optic-disc exclusion (od_run1_b0) at full resolution, any-pixel downsample to 512 (INTER_AREA > 0), fundus clip at 512 (`clip_region`, erode 6 px). The 512 control is what ControlNet sees, so 512 is the **primary** level; full resolution is reported for comparison with the record.

**Data (fixed in `inputs/detector_eval_manifest.csv`, created once):**
- *Diseased:* IDRiD-test (27; test only, and note it was used to pick both checkpoints' epochs, so it flatters them); DDR-test (225, never seen by either detector), split once at random, stratified by EX/HE presence, into a **validation half** (all tuning) and a **test half** (confirmation only).
- *Healthy:* EyePACS grade 0, one eye per patient, from the patient split: **holdout -> test** (300), **dev -> validation** (150); the 9 healthy holdout-test patients of the ControlNet set excluded.

**Measures (512 level).** Lesion = 4-connected GT region; *found* = a detection within 2 px (as `healthy_eye_check_v2.tol_recall`). Recall pooled over lesions (bootstrap CI by image). Precision strict (detected px on GT) and 2 px tolerant, pooled and per image. Healthy eyes: false-positive px per eye, % of eyes with any detection (Wilson CI), detected blobs per eye.

**Stage 1 questions.** D1 how well do the detectors find grader lesions (IDRiD-test, DDR test half), including the first DDR measurement of EX; D2 how much do they fire on healthy eyes; D3 where are the false positives (disc / rim / fovea / other, by rule); D4 what goes with misses (lesion size, image scale, brightness, lesions per image).

**Stage 2 candidates (no retraining).** Threshold (grid); 3-seed ensemble; flip test-time augmentation; size normalisation (photo resized so the fundus width matches the IDRiD-train median); Gaussian-weighted tile blending; extra rim margin; minimum blob size; HE disc exclusion; HE fovea exclusion (heuristic fovea). The 512-upsampled protocol (used by the automated reader) is measured for reference, not as a candidate.

**Selection rule (Stage 2, validation data only).** A configuration qualifies only if, on validation, its healthy false-positive px per eye is no higher than the baseline's and its tolerant pooled precision on DDR-val is no lower. Among qualifiers, pick the highest DDR-val recall; ties go to fewer healthy false positives. If nothing qualifies, the baseline stays and the trade-off table goes to Adam.

**Confirmation (once, test data).** Chosen vs baseline on the same images: lesion-level McNemar (exact), per-image Wilcoxon on false-positive px (diseased and healthy), McNemar on healthy any-detection; Holm across the family.

**Known limits.** DDR has no patient IDs (images treated as independent). EyePACS grade 0 labels are imperfect, so some "false positives" may be real lesions: Victoria's check of rule-picked cases (section 11) settles that for the cases shown. The fovea location is a heuristic (darkest smoothed area away from the disc).

## Parameters

In [ ]:
from pathlib import Path
import os, json, time, hashlib, shutil, warnings
from decimal import Decimal, ROUND_HALF_UP
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt, torch
from scipy import ndimage, stats
from skimage import measure as sk_measure
from skimage.morphology import dilation as sk_dilation
from IPython.display import display
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 120)

ROOT = Path(os.environ.get("RETINA_ROOT", Path.cwd().parent)).resolve()   # project root (notebook lives in ROOT/notebooks)
assert (ROOT / "notebooks").is_dir(), ROOT
INPUTS = ROOT / "notebooks" / "inputs"; INPUTS.mkdir(parents=True, exist_ok=True)
OUT = ROOT / "notebooks" / "outputs" / "detector_evaluation"; OUT.mkdir(parents=True, exist_ok=True)
CACHE = OUT / "cache"
MANIFEST, META = INPUTS / "detector_eval_manifest.csv", INPUTS / "detector_eval_meta.json"

SUBSET = None            # list of image names for a quick test run; disables the record reproduction and the claims
RUN_INFERENCE = True     # compute missing cache entries (GPU recommended)
MAKE_PACKAGE = False     # local only: copy the manifest's files + models + notebook into PKG_DIR for RunPod
PKG_DIR = ROOT / "detector_eval_pkg"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH = 8 if DEVICE == "cuda" else 2

BASE_THR = 0.35
THR_GRID = [0.2, 0.3, 0.35, 0.4, 0.5, 0.6, 0.7, 0.8]
WINDOW = {"ex": (512, 384), "he": (768, 640)}       # patch, stride (decision.md L852; record validation)
CKPTS = {  # path, expected epoch, expected recall (decision.md L356-363, L755-757), md5 prefix (verified 2026-11-03/04, local == Drive)
    ("ex", "s42"): ("models/ex_detector/ex_detector_best.pth", 5, 0.7638, "0c342b8a2d"),
    ("ex", "s123"): ("models/ex_run1_seed123/ex_detector_best.pth", 15, 0.7573, "65c6189790"),
    ("ex", "s7"): ("models/ex_run1_seed7/ex_detector_best.pth", 5, 0.7638, "d806b162d5"),
    ("he", "s42"): ("models/he_detector/he_detector_best.pth", 35, 0.7084, "76b659c647"),
    ("he", "s123"): ("models/he_run5_seed123_v2/he_detector_best.pth", 35, 0.7047, "ed234632c6"),
    ("he", "s7"): ("models/he_run5_seed7/he_detector_best.pth", 35, 0.6879, "41a6c5b1a2")}
OD_MD5 = "743f1aa019"
B2_BYTES = 40_673_694                                # efficientnet-b2 U-Net checkpoint size (decision.md L440)
OD_CKPT = "models/od_detector/od_detector_best.pth"
VARIANTS = ["s42", "s123", "s7", "tta", "ens3", "scale", "gauss", "up512"]
CANDIDATES = [v for v in VARIANTS if v != "up512"]
N_HEALTHY_TEST, N_HEALTHY_VAL = 300, 150
EXCLUDED_PATIENTS = [10977, 10983, 10987, 10988, 10989, 1000, 10003, 10007, 10009]   # ControlNet healthy test patients
TOL_PX, CLIP_ERODE = 2, 6
SEED, N_BOOT, ALPHA = 0, 10_000, 0.05
K_TOL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * TOL_PX + 1, 2 * TOL_PX + 1))
rhu = lambda x, nd=0: (lambda q: str(q.copy_abs() if q == 0 else q))(Decimal(str(float(x))).quantize(Decimal(1).scaleb(-nd), rounding=ROUND_HALF_UP))
rel = lambda p: Path(p).resolve().relative_to(ROOT).as_posix()
print(ROOT, DEVICE)

## 1. Fixed manifest (created once, never overwritten)

In [ ]:
def tissue_mask(img_rgb, thr=20):                   # verbatim from scratch/vessel_clip_proto.py
    t = (img_rgb.max(axis=2) > thr).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(t)
    if n <= 1:
        return t > 0
    t = (lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
    cs, _ = cv2.findContours(t, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    filled = np.zeros_like(t); cv2.drawContours(filled, cs, -1, 1, -1)
    return filled > 0
def clip_region(img_rgb, erode=CLIP_ERODE):          # verbatim from scratch/vessel_clip_proto.py
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * erode + 1, 2 * erode + 1))
    return cv2.erode(tissue_mask(img_rgb).astype(np.uint8), k) > 0
def fov_width(bgr):
    f = 8; small = cv2.resize(bgr, (bgr.shape[1] // f, bgr.shape[0] // f), interpolation=cv2.INTER_AREA)
    xs = np.nonzero(tissue_mask(cv2.cvtColor(small, cv2.COLOR_BGR2RGB)).any(0))[0]
    return float((xs.max() - xs.min() + 1) * f)
def read_gt(p):
    g = cv2.imread(str(ROOT / p), cv2.IMREAD_GRAYSCALE); assert g is not None, p; return g > 0

if not MANIFEST.exists():
    rows, idr = [], ROOT / "A. Segmentation" / "A. Segmentation"
    gtd = idr / "2. All Segmentation Groundtruths" / "b. Testing Set"
    for p in sorted((idr / "1. Original Images" / "b. Testing Set").glob("*.jpg")):
        ex, he = next((gtd / "3. Hard Exudates").glob(f"{p.stem}_EX.*"), None), next((gtd / "2. Haemorrhages").glob(f"{p.stem}_HE.*"), None)
        rows.append(dict(name=p.stem, group="IDRiD", role="test", image=rel(p), gt_ex=rel(ex) if ex else "", gt_he=rel(he) if he else "", patient=""))
    ddr = ROOT / "scratch" / "ddr_lesion" / "DDR-dataset" / "lesion_segmentation" / "test"
    d = []
    for p in sorted((ddr / "image").glob("*.jpg")):
        ex, he = ddr / "label" / "EX" / f"{p.stem}.tif", ddr / "label" / "HE" / f"{p.stem}.tif"
        assert ex.exists() and he.exists(), p.stem
        d.append(dict(name=p.stem, group="DDR", image=rel(p), gt_ex=rel(ex), gt_he=rel(he), patient="", s=(read_gt(rel(ex)).any(), read_gt(rel(he)).any())))
    rs = np.random.default_rng(SEED)
    for s in sorted({x["s"] for x in d}):
        idx = [i for i, x in enumerate(d) if x["s"] == s]; idx = [idx[i] for i in rs.permutation(len(idx))]
        for j, i in enumerate(idx): d[i]["role"] = "val" if j % 2 == 0 else "test"
    rows += [{k: v for k, v in x.items() if k != "s"} for x in d]
    split = pd.read_csv(ROOT / "scratch" / "patient_split.csv")
    local = {p.stem: p for p in (ROOT / "eyepacs_from_zip").rglob("*.jp*g")}   # graded EyePACS train photos (extracted from gdrive eyepacs/train.zip.001)
    for part, n_take, role in (("holdout", N_HEALTHY_TEST, "test"), ("dev", N_HEALTHY_VAL, "val")):
        s = split[(split.split == part) & (split.level == 0) & split.image.isin(local) & ~split.patient_id.isin(EXCLUDED_PATIENTS)]
        s = s.groupby("patient_id").sample(1, random_state=SEED).sample(n_take, random_state=SEED).sort_values("image")
        rows += [dict(name=r.image, group="healthy", role=role, image=rel(local[r.image]), gt_ex="", gt_he="", patient=str(r.patient_id)) for r in s.itertuples()]
    trainp = sorted((idr / "1. Original Images" / "a. Training Set").glob("*.jpg"))
    meta = dict(D_REF=float(np.median([fov_width(cv2.imread(str(p))) for p in trainp])), n_idrid_train_for_dref=len(trainp),
                seed=SEED, created=time.strftime("%Y-%m-%d %H:%M"))
    pd.DataFrame(rows).to_csv(MANIFEST, index=False); META.write_text(json.dumps(meta, indent=1))
man = pd.read_csv(MANIFEST, dtype=str).fillna("")
MD5F = INPUTS / "detector_eval_md5.csv"
if not MD5F.exists():                                # recorded once where the original files are (local), verified everywhere else
    paths = sorted({p for c in ("image", "gt_ex", "gt_he") for p in man[c] if p})
    pd.DataFrame({"path": paths, "md5": [hashlib.md5((ROOT / p).read_bytes()).hexdigest() for p in paths]}).to_csv(MD5F, index=False)
    man[man.group == "healthy"].image.map(lambda p: Path(p).name).to_csv(INPUTS / "eyepacs_selected.txt", index=False, header=False)
meta = json.loads(META.read_text()); D_REF = meta["D_REF"]
if SUBSET: man = man[man.name.isin(SUBSET)].reset_index(drop=True); print("SUBSET MODE:", len(man), "images; record reproduction and claims disabled")
display(man.groupby(["group", "role"]).size().rename("images").to_frame()); print("D_REF (IDRiD-train median fundus width, px):", D_REF)

## 2. Integrity checks

In [ ]:
missing = [p for c in ("image", "gt_ex", "gt_he") for p in man[c] if p and not (ROOT / p).exists()]
assert not missing, f"{len(missing)} files missing, e.g. {missing[:3]} (on the pod: assemble the data first, see the RunPod steps)"
want = pd.read_csv(MD5F).set_index("path").md5
paths = {p for c in ("image", "gt_ex", "gt_he") for p in man[c] if p}
bad = [p for p in sorted(paths) if hashlib.md5((ROOT / p).read_bytes()).hexdigest() != want[p]]
assert not bad, f"{len(bad)} files differ from the recorded MD5, e.g. {bad[:3]}"
print(f"MD5 verified for {len(paths)} files")
full = pd.read_csv(MANIFEST, dtype=str).fillna("")
assert full.name.is_unique
h = full[full.group == "healthy"]
assert not h.patient.astype(int).isin(EXCLUDED_PATIENTS).any()
assert set(h[h.role == "test"].patient).isdisjoint(set(h[h.role == "val"].patient)), "healthy test and val share a patient"
assert h.groupby("role").patient.apply(lambda s: s.is_unique).all(), "more than one eye per patient"
d = full[full.group == "DDR"]; assert set(d[d.role == "val"].name).isdisjoint(d[d.role == "test"].name) and len(d) == 225
info = []
for r in man.itertuples():
    bgr = cv2.imread(str(ROOT / r.image)); assert bgr is not None, r.image
    row = dict(name=r.name, group=r.group, role=r.role, H=bgr.shape[0], W=bgr.shape[1], fov_w=fov_width(bgr))
    for det in ("ex", "he"):
        p = getattr(r, f"gt_{det}")
        if p:
            g = read_gt(p); assert g.shape == bgr.shape[:2], (r.name, det); row[f"gt_{det}_px"] = int(g.sum())
    info.append(row)
INFO = pd.DataFrame(info).set_index("name"); INFO["scale"] = D_REF / INFO.fov_w
print("all files present; GT shapes match images; patient-level separation holds")
display(INFO.groupby(["group", "role"])[["W", "H", "fov_w", "scale"]].describe().T.round(2))

## 2b. Package for RunPod (local only; run cells up to here with MAKE_PACKAGE = True)

In [ ]:
if MAKE_PACKAGE:
    files = set(p for c in ("image", "gt_ex", "gt_he") for p in pd.read_csv(MANIFEST, dtype=str).fillna("")[c] if p)
    files |= {v[0] for v in CKPTS.values()} | {OD_CKPT}
    files |= {rel(MANIFEST), rel(META), "notebooks/detector_evaluation.ipynb"}
    total = 0
    for f in sorted(files):
        dst = PKG_DIR / f; dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(ROOT / f, dst); total += dst.stat().st_size
    print(f"package: {len(files)} files, {total / 1e9:.2f} GB in {PKG_DIR}")

## 3. Models (identity asserted) and inference functions

In [ ]:
import segmentation_models_pytorch as smp
def load_ckpt(path, epoch, recall, md5):
    p = ROOT / path; assert p.stat().st_size == B2_BYTES, f"{path}: not a b2 checkpoint ({p.stat().st_size} bytes)"
    c = torch.load(p, map_location=DEVICE, weights_only=False)
    assert c["epoch"] == epoch and round(float(c["recall"]), 4) == recall, f"{path}: epoch {c['epoch']} recall {c['recall']}"
    assert float(c.get("threshold", 0.35)) == BASE_THR
    m = smp.Unet(encoder_name="efficientnet-b2", encoder_weights=None, in_channels=3, classes=1)
    m.load_state_dict(c["state_dict"]); m.to(DEVICE).eval()
    h = hashlib.md5(p.read_bytes()).hexdigest(); assert h.startswith(md5), f"{path}: md5 {h[:10]} != {md5}"
    return m, h
MODELS, MD5 = {}, {}
for k, (p, e, r, h) in CKPTS.items(): MODELS[k], MD5["/".join(k)] = load_ckpt(p, e, r, h)
oc = torch.load(ROOT / OD_CKPT, map_location=DEVICE, weights_only=False)
assert oc.get("encoder") == "efficientnet-b0" and oc.get("epoch") == 5 and round(oc.get("dice", 0), 3) == 0.963, "not od_run1_b0"
OD = smp.Unet(encoder_name="efficientnet-b0", encoder_weights=None, in_channels=3, classes=1); OD.load_state_dict(oc["state_dict"]); OD.to(DEVICE).eval()
OD_SIZE = oc["img_size"]; MD5["od"] = hashlib.md5((ROOT / OD_CKPT).read_bytes()).hexdigest(); assert MD5["od"].startswith(OD_MD5)
display(pd.Series(MD5, name="md5"))

def preprocess(bgr):                                 # as generate_controlnet_masks.preprocess_for_detector / training
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB); l, a, b = cv2.split(cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB))
    return cv2.cvtColor(cv2.merge([cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l), a, b]), cv2.COLOR_LAB2RGB)
def starts(L, patch, stride):
    return sorted({max(0, min(s, L - patch)) for s in list(range(0, L - patch, stride)) + [L - patch]})
GW = {}
def gauss_w(patch):
    if patch not in GW:
        g = np.exp(-0.5 * ((np.arange(patch) - (patch - 1) / 2) / (patch / 4)) ** 2); GW[patch] = np.maximum(np.outer(g, g), 1e-3).astype(np.float32)
    return GW[patch]
def infer(model, rgb, det, flip=False):
    patch, stride = WINDOW[det]
    if flip: rgb = rgb[:, ::-1]
    H, W = rgb.shape[:2]; pad = ((0, max(0, patch - H)), (0, max(0, patch - W)), (0, 0))
    x = np.ascontiguousarray(np.pad(rgb, pad)); Hp, Wp = x.shape[:2]
    acc, cnt = np.zeros((Hp, Wp), np.float32), np.zeros((Hp, Wp), np.float32); accg, cntg = acc.copy(), cnt.copy(); w = gauss_w(patch)
    coords = [(y, xx) for y in starts(Hp, patch, stride) for xx in starts(Wp, patch, stride)]
    with torch.no_grad():
        for i in range(0, len(coords), BATCH):
            cb = coords[i:i + BATCH]
            t = torch.from_numpy(np.stack([x[y:y + patch, xx:xx + patch] for y, xx in cb]).transpose(0, 3, 1, 2).copy()).float().div(255).to(DEVICE)
            pr = torch.sigmoid(model(t))[:, 0].cpu().numpy()
            for (y, xx), p in zip(cb, pr):
                acc[y:y + patch, xx:xx + patch] += p; cnt[y:y + patch, xx:xx + patch] += 1
                accg[y:y + patch, xx:xx + patch] += p * w; cntg[y:y + patch, xx:xx + patch] += w
    pu, pg = (acc / np.maximum(cnt, 1))[:H, :W], (accg / np.maximum(cntg, 1e-6))[:H, :W]
    if flip: pu, pg = pu[:, ::-1], pg[:, ::-1]
    return np.ascontiguousarray(pu), np.ascontiguousarray(pg)
def od_full(bgr):                                    # as generate_controlnet_masks.od_mask_fullres
    rgb = preprocess(bgr); H, W = rgb.shape[:2]; r = cv2.resize(rgb, (OD_SIZE, OD_SIZE))
    with torch.no_grad():
        p = torch.sigmoid(OD(torch.from_numpy(r.transpose(2, 0, 1).copy()).float().unsqueeze(0).div(255).to(DEVICE)))[0, 0].cpu().numpy()
    return cv2.resize((p > 0.5).astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST) > 0
to512 = lambda m: cv2.resize(m.astype(np.uint8) * 255, (512, 512), interpolation=cv2.INTER_AREA) > 0     # any-pixel, as production
print("models loaded on", DEVICE)

## 4. Inference cache
Per image: optic-disc mask, 512 RGB, the full-resolution seed-42 probability map for each detector (for the record reproduction), and for every variant the 512-level masks at each threshold of the grid. EX masks carry the production optic-disc exclusion. Keyed to the checkpoint MD5s; a rerun only computes what is missing.

In [ ]:
for sub in ("grid", "prob", "od", "rgb512"): (CACHE / sub).mkdir(parents=True, exist_ok=True)
CM = CACHE / "manifest.json"
cache_key = dict(md5=MD5, thr=THR_GRID, window=WINDOW, D_REF=D_REF, variants=VARIANTS, version=1)
if CM.exists(): assert json.loads(CM.read_text()) == json.loads(json.dumps(cache_key)), "cache built with different settings: delete outputs/detector_evaluation/cache"
else: CM.write_text(json.dumps(cache_key))
gpath = lambda n, det: CACHE / "grid" / f"{n}__{det}.npz"
ppath = lambda n, det: CACHE / "prob" / f"{n}__{det}.png"
def save_prob(p, path): cv2.imwrite(str(path), np.where(p < 0.02, 0, np.round(p * 65535)).astype(np.uint16))
def load_prob(path): return cv2.imread(str(path), cv2.IMREAD_UNCHANGED).astype(np.float32) / 65535
def run_image(r):
    bgr = cv2.imread(str(ROOT / r.image)); H, W = bgr.shape[:2]
    disc = od_full(bgr); cv2.imwrite(str(CACHE / "od" / f"{r.name}.png"), disc.astype(np.uint8) * 255)
    b512 = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA); cv2.imwrite(str(CACHE / "rgb512" / f"{r.name}.png"), b512)
    rgb = preprocess(bgr); s = float(np.clip(D_REF / INFO.loc[r.name, "fov_w"], 0.5, 2.5))
    bs = cv2.resize(bgr, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC if s > 1 else cv2.INTER_AREA)
    up = cv2.resize(b512, (W, H), interpolation=cv2.INTER_CUBIC)
    for det in ("ex", "he"):
        if gpath(r.name, det).exists() and ppath(r.name, det).exists(): continue
        p42, g42 = infer(MODELS[(det, "s42")], rgb, det); save_prob(p42, ppath(r.name, det))
        pf = infer(MODELS[(det, "s42")], rgb, det, flip=True)[0]
        p123, p7 = infer(MODELS[(det, "s123")], rgb, det)[0], infer(MODELS[(det, "s7")], rgb, det)[0]
        ps = cv2.resize(infer(MODELS[(det, "s42")], preprocess(bs), det)[0], (W, H), interpolation=cv2.INTER_LINEAR)
        pu = infer(MODELS[(det, "s42")], preprocess(up), det)[0]
        P = dict(s42=p42, s123=p123, s7=p7, tta=(p42 + pf) / 2, ens3=(p42 + p123 + p7) / 3, scale=ps, gauss=g42, up512=pu)
        out = {}
        for k, p in P.items():
            if det == "ex": p = np.where(disc, 0, p)
            out[k] = np.packbits(np.stack([to512(p > t) for t in THR_GRID]).reshape(len(THR_GRID), -1), axis=1)
        np.savez_compressed(gpath(r.name, det), **out)
todo = [r for r in man.itertuples() if not (gpath(r.name, "ex").exists() and gpath(r.name, "he").exists() and ppath(r.name, "ex").exists() and ppath(r.name, "he").exists())]
print(f"{len(todo)} of {len(man)} images to run on {DEVICE}"); t0 = time.time()
if RUN_INFERENCE:
    for i, r in enumerate(todo):
        run_image(r)
        if i % 10 == 0 or i == len(todo) - 1: print(f"  {i + 1}/{len(todo)} {r.name} ({time.time() - t0:.0f}s)", flush=True)
assert all(gpath(n, d).exists() for n in man.name for d in ("ex", "he")), "cache incomplete: run inference (RUN_INFERENCE = True)"
def grids(n, det):                                   # packed (bits); unpacked one mask at a time in mask_for
    z = np.load(gpath(n, det)); return {k: z[k] for k in z.files}
print("cache complete")

## 5. Record reproduction (must pass before anything new counts)
Full resolution, seed 42, threshold 0.35, the training scripts' own metric functions (`recall_at_iou`, `pixel_preservation`, copied verbatim from `train_he_detector.py`). Targets from decision.md: HE IDRiD-test recall 0.7084, preservation 0.6957 (L481-482); HE DDR-test (194 images with HE) 1,957 / 6,457 blobs, mean pixel recall 0.572, pooled precision 0.506, 32 images with pixel recall < 0.3 (L1679-1684); EX IDRiD-test recall 0.7638 (L361), pooled precision raw 0.7688 / disc-excluded 0.7689, per-image 0.6549 / 0.6551 (L1066-1067).

In [ ]:
def pixel_preservation(pred_bin, gt_bin):          # verbatim
    total = int(gt_bin.sum())
    if total == 0:
        return None
    return float((gt_bin & pred_bin).sum()) / total
def recall_at_iou(pred_bin, gt_bin, iou_thresh=0.3):  # verbatim
    gt_labeled = sk_measure.label(gt_bin); pred_labeled = sk_measure.label(pred_bin)
    gt_regions = sk_measure.regionprops(gt_labeled); pred_areas = {r.label: r.area for r in sk_measure.regionprops(pred_labeled)}
    if not gt_regions:
        return None, 0, 0
    hits = 0
    for gt_reg in gt_regions:
        r0, c0, r1, c1 = gt_reg.bbox
        gt_blob = (gt_labeled[r0:r1, c0:c1] == gt_reg.label).astype(np.uint8); pred_crop = pred_labeled[r0:r1, c0:c1]
        overlap = np.unique(pred_crop[gt_blob > 0]); overlap = overlap[overlap > 0]; best_iou = 0.0
        for pl in overlap:
            inter = (gt_blob & (pred_crop == pl)).sum(); union = gt_reg.area + pred_areas.get(pl, 0) - inter; best_iou = max(best_iou, inter / max(union, 1))
        if best_iou >= iou_thresh:
            hits += 1
    return hits / len(gt_regions), hits, len(gt_regions)
FR = []
for r in man[man.group != "healthy"].itertuples():
    disc = cv2.imread(str(CACHE / "od" / f"{r.name}.png"), cv2.IMREAD_GRAYSCALE) > 0
    for det in ("ex", "he"):
        gp = getattr(r, f"gt_{det}")
        if not gp: continue
        gt = read_gt(gp).astype(np.uint8)
        if not gt.any(): continue
        raw = (load_prob(ppath(r.name, det)) > BASE_THR).astype(np.uint8)
        for excl, pred in (("raw", raw), ("disc_excluded", (raw.astype(bool) & ~disc).astype(np.uint8))):
            if det == "he" and excl == "disc_excluded": continue
            _, hits, n = recall_at_iou(pred, gt)
            FR.append(dict(name=r.name, group=r.group, role=r.role, det=det, mode=excl, hits=hits, blobs=n, pres=pixel_preservation(pred.astype(bool), gt.astype(bool)),
                           tp=int((gt.astype(bool) & pred.astype(bool)).sum()), pred_px=int(pred.sum())))
FR = pd.DataFrame(FR); FR.to_csv(OUT / "fullres_record_metrics.csv", index=False)
def rec(df): return dict(images=len(df), recall=df.hits.sum() / df.blobs.sum(), hits=int(df.hits.sum()), blobs=int(df.blobs.sum()), mean_pres=df.pres.mean(),
                         prec_pooled=df.tp.sum() / max(df.pred_px.sum(), 1), prec_mean=(df.tp / df.pred_px.clip(lower=1)).mean(), low_pres=int((df.pres < 0.3).sum()))
R = {k: rec(FR[(FR.group == g) & (FR.det == d) & (FR["mode"] == m)]) for k, (g, d, m) in
     {"he_idrid": ("IDRiD", "he", "raw"), "he_ddr": ("DDR", "he", "raw"), "ex_idrid_raw": ("IDRiD", "ex", "raw"), "ex_idrid_disc": ("IDRiD", "ex", "disc_excluded"),
      "ex_ddr_disc": ("DDR", "ex", "disc_excluded")}.items() if len(FR[(FR.group == g) & (FR.det == d)])}
display(pd.DataFrame(R).T.round(4))
if not SUBSET:
    chk = [("he_idrid", "recall", 0.7084, 0.002), ("he_idrid", "mean_pres", 0.6957, 0.003), ("he_ddr", "images", 194, 0), ("he_ddr", "blobs", 6457, 0),
           ("he_ddr", "hits", 1957, 15), ("he_ddr", "mean_pres", 0.572, 0.003), ("he_ddr", "prec_pooled", 0.506, 0.003), ("he_ddr", "low_pres", 32, 2),
           ("ex_idrid_raw", "recall", 0.7638, 0.002), ("ex_idrid_raw", "prec_pooled", 0.7688, 0.002), ("ex_idrid_raw", "prec_mean", 0.6549, 0.003),
           ("ex_idrid_disc", "prec_pooled", 0.7689, 0.002), ("ex_idrid_disc", "prec_mean", 0.6551, 0.003)]
    res = pd.DataFrame([dict(check=f"{k}.{m}", record=v, now=R[k][m], ok=abs(R[k][m] - v) <= tol) for k, m, v, tol in chk]); display(res)
    assert res.ok.all(), "record not reproduced: stop and resolve before using any new number"
    print("Record reproduced.")

## 6. Data exploration
Image and fundus sizes, lesion sizes at native and 512 resolution, brightness, per source. This is where the size hypothesis (DDR lesions smaller in pixels) is looked at before any fix is tested.

In [ ]:
RGB512 = {n: cv2.cvtColor(cv2.imread(str(CACHE / "rgb512" / f"{n}.png")), cv2.COLOR_BGR2RGB) for n in man.name}
OD512 = {n: to512(cv2.imread(str(CACHE / "od" / f"{n}.png"), cv2.IMREAD_GRAYSCALE) > 0) for n in man.name}
CLIP = {n: clip_region(RGB512[n]) for n in man.name}
TISSUE = {n: tissue_mask(RGB512[n]) for n in man.name}
GT512 = {}
for r in man.itertuples():
    for det in ("ex", "he"):
        p = getattr(r, f"gt_{det}"); GT512[(r.name, det)] = to512(read_gt(p)) if p else np.zeros((512, 512), bool)
def fovea(n):                                        # heuristic: darkest smoothed green away from the disc, inside the eroded fundus
    g = cv2.GaussianBlur(RGB512[n][..., 1].astype(np.float32), (0, 0), 8)
    allowed = cv2.erode(TISSUE[n].astype(np.uint8), np.ones((41, 41), np.uint8)) > 0
    allowed &= ~(cv2.dilate(OD512[n].astype(np.uint8), np.ones((81, 81), np.uint8)) > 0)
    if not allowed.any(): return None
    y, x = np.unravel_index(np.argmin(np.where(allowed, g, np.inf)), g.shape)
    w = TISSUE[n].any(0).sum(); return int(y), int(x), max(4, int(round(0.07 * w)))
FOVEA = {n: fovea(n) for n in man.name}
les = []
for r in man[man.group != "healthy"].itertuples():
    for det in ("ex", "he"):
        g = GT512[(r.name, det)]
        if not g.any(): continue
        k, lab, st, _ = cv2.connectedComponentsWithStats(g.astype(np.uint8), connectivity=4)
        les += [dict(name=r.name, group=r.group, role=r.role, det=det, area512=int(a)) for a in st[1:, cv2.CC_STAT_AREA]]
LES = pd.DataFrame(les)
INFO["brightness"] = [RGB512[n][..., 1][TISSUE[n]].mean() for n in INFO.index]
INFO["contrast"] = [RGB512[n][..., 1][TISSUE[n]].std() for n in INFO.index]
display(INFO.groupby("group")[["fov_w", "scale", "brightness", "contrast"]].median().round(2))
display(LES.groupby(["det", "group"]).area512.describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(1))
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for g, col in zip(["IDRiD", "DDR", "healthy"], ["#2a6f97", "#c0504d", "#5b8c5a"]):
    ax[0].hist(INFO[INFO.group == g].fov_w, bins=20, alpha=0.6, color=col, label=g)
    for j, det in enumerate(("ex", "he")):
        l = LES[(LES.group == g) & (LES.det == det)]
        if len(l): ax[1 + j].hist(np.log10(l.area512), bins=30, alpha=0.6, color=col, label=f"{g} ({len(l)} lesions)", density=True)
ax[0].axvline(D_REF, color="k", ls="--", lw=1); ax[0].set(xlabel="fundus width, native px", ylabel="images", title="Fundus size (dashed: IDRiD-train median)")
for j, det in enumerate(("EX", "HE")): ax[1 + j].set(xlabel="log10 GT lesion area at 512 (px)", ylabel="density", title=f"{det} lesion size at 512")
for a in ax: a.legend(frameon=False)
fig.tight_layout(); fig.savefig(OUT / "eda.png", dpi=110); plt.show()

## 7. Scoring at 512 (three independent matching paths)

In [ ]:
def post(mask, n, det, rim=0, min_size=1, disc=False, fov=False):
    m = mask & CLIP[n]
    if rim: m &= cv2.erode(TISSUE[n].astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * (CLIP_ERODE + rim) + 1,) * 2)) > 0
    if disc: m &= ~(cv2.dilate(OD512[n].astype(np.uint8), np.ones((7, 7), np.uint8)) > 0)
    if fov and FOVEA[n] is not None:
        y, x, rad = FOVEA[n]; yy, xx = np.ogrid[:512, :512]; m &= (yy - y) ** 2 + (xx - x) ** 2 > rad ** 2
    if min_size > 1:
        k, lab, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=4); keep = np.r_[False, st[1:, cv2.CC_STAT_AREA] >= min_size]; m = keep[lab]
    return m
LAB = {}
def gt_lab(n, det):
    if (n, det) not in LAB: LAB[(n, det)] = cv2.connectedComponents(GT512[(n, det)].astype(np.uint8), connectivity=4)
    return LAB[(n, det)]
def score_A(m, n, det):                              # cv2 path
    g = GT512[(n, det)]; k, lab = gt_lab(n, det); md = cv2.dilate(m.astype(np.uint8), K_TOL) > 0
    found = np.zeros(k, bool); found[np.unique(lab[md & (lab > 0)])] = True
    gd = cv2.dilate(g.astype(np.uint8), K_TOL) > 0
    kp, lp = cv2.connectedComponents(m.astype(np.uint8), connectivity=4)
    fp_blobs = len(set(range(1, kp)) - set(np.unique(lp[gd & (lp > 0)])))
    return dict(lesions=k - 1, found=int(found[1:].sum()), pred_px=int(m.sum()), tp_strict=int((m & g).sum()), tp_tol=int((m & gd).sum()), fp_px=int((m & ~gd).sum()),
                blobs=kp - 1, fp_blobs=fp_blobs), found[1:]
def score_B(m, n, det):                              # scipy path
    g = GT512[(n, det)]; lab, k = ndimage.label(g); fk = K_TOL.astype(bool); md = ndimage.binary_dilation(m, structure=fk); gd = ndimage.binary_dilation(g, structure=fk)
    found = np.array([md[lab == i].any() for i in range(1, k + 1)], bool)
    return dict(lesions=k, found=int(found.sum()), pred_px=int(np.count_nonzero(m)), tp_strict=int(np.count_nonzero(m & g)), tp_tol=int(np.count_nonzero(m & gd))), found
def score_C(m, n, det):                              # skimage path
    g = GT512[(n, det)]; lab = sk_measure.label(g, connectivity=1); fk = K_TOL.astype(bool); md = sk_dilation(m, footprint=fk); gd = sk_dilation(g, footprint=fk)
    found = np.zeros(lab.max() + 1, bool); found[np.unique(lab[md & (lab > 0)])] = True
    return dict(lesions=int(lab.max()), found=int(found[1:].sum()), pred_px=int(m.sum()), tp_strict=int(ndimage.sum(m, labels=g.astype(int), index=1)) if g.any() else 0,
                tp_tol=int((m & gd).sum())), found[1:]
BASE = dict(variant="s42", thr=BASE_THR, rim=0, min_size=1, disc=False, fov=False)
G = {}
def mask_for(n, det, cfg):
    if (n, det) not in G: G[(n, det)] = grids(n, det)
    m = np.unpackbits(G[(n, det)][cfg["variant"]][THR_GRID.index(cfg["thr"])])[:512 * 512].reshape(512, 512).astype(bool)
    return post(m, n, det, cfg["rim"], cfg["min_size"], cfg["disc"], cfg["fov"])
def evaluate(cfg, names, det, paths=False):
    rows, flags = [], []
    for n in names:
        m = mask_for(n, det, cfg); a, fa = score_A(m, n, det)
        if paths:
            b, fb = score_B(m, n, det); c, fc = score_C(m, n, det)
            assert all(a[k] == b[k] == c[k] for k in b), (n, det, a, b, c); assert np.array_equal(fa, fb) and np.array_equal(fa, fc), (n, det)
        rows.append(dict(name=n, **a)); flags.append(fa)
    return pd.DataFrame(rows), flags
ROLE = {k: list(v) for k, v in man.groupby(["group", "role"]).name}
for det in ("ex", "he"):
    for key, names in ROLE.items(): evaluate(BASE, names, det, paths=True)
print("three matching paths agree on every image, both detectors, baseline configuration")

## 8. Stage 1 results: the production detectors

In [ ]:
rng = np.random.default_rng(SEED)
def wilson(k, n): ci = stats.binomtest(int(k), int(n)).proportion_ci(0.95, method="wilson") if n else None; return (100 * ci.low, 100 * ci.high) if ci else (np.nan, np.nan)
def boot(df, num, den, scale=100):
    a = df[[num, den]].to_numpy(float); idx = rng.integers(0, len(a), (N_BOOT, len(a))); s = a[idx].sum(1)
    with np.errstate(invalid="ignore", divide="ignore"): return np.nanpercentile(scale * s[:, 0] / s[:, 1], [2.5, 97.5])
def summarise(df, group):
    if group == "healthy":
        k = int((df.pred_px > 0).sum()); lo, hi = wilson(k, len(df)); b = df.fp_px.to_numpy(float); idx = rng.integers(0, len(b), (N_BOOT, len(b)))
        bl, bh = np.percentile(b[idx].mean(1), [2.5, 97.5])
        return dict(images=len(df), fp_px_per_eye=b.mean(), fp_px_ci_low=bl, fp_px_ci_high=bh, eyes_with_any_pct=100 * k / len(df), any_ci_low=lo, any_ci_high=hi, blobs_per_eye=df.blobs.mean())
    d = df[df.lesions > 0]; lo, hi = boot(d, "found", "lesions"); plo, phi = boot(df[df.pred_px > 0], "tp_tol", "pred_px")
    return dict(images=len(d), lesions=int(d.lesions.sum()), recall_pct=100 * d.found.sum() / d.lesions.sum(), recall_ci_low=lo, recall_ci_high=hi,
                prec_tol_pct=100 * df.tp_tol.sum() / max(df.pred_px.sum(), 1), prec_tol_ci_low=plo, prec_tol_ci_high=phi,
                prec_strict_pct=100 * df.tp_strict.sum() / max(df.pred_px.sum(), 1), prec_tol_per_image_pct=100 * (df.tp_tol / df.pred_px)[df.pred_px > 0].mean(),
                fp_px_per_image=df.fp_px.mean())
S1, EV = [], {}
for det in ("ex", "he"):
    for (g, role), names in ROLE.items():
        df, fl = evaluate(BASE, names, det); EV[(det, g, role)] = (df, fl); S1.append(dict(det=det, group=g, role=role, **summarise(df, g)))
S1 = pd.DataFrame(S1); display(S1.round(2)); S1.to_csv(OUT / "stage1_baseline.csv", index=False)

### 8a. Where the false positives are (rule-based classes) and what goes with misses

In [ ]:
def fp_classes(n, det, m):
    gd = cv2.dilate(GT512[(n, det)].astype(np.uint8), K_TOL) > 0
    k, lab, st, cen = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=4)
    dist_edge = cv2.distanceTransform(TISSUE[n].astype(np.uint8), cv2.DIST_L2, 5); odd = cv2.dilate(OD512[n].astype(np.uint8), np.ones((21, 21), np.uint8)) > 0
    out = []
    for j in range(1, k):
        if gd[lab == j].any(): continue
        x, y = int(round(cen[j][0])), int(round(cen[j][1])); fv = FOVEA[n]
        cls = ("disc" if odd[y, x] else "rim" if dist_edge[y, x] < 15 else
               "fovea (heuristic)" if fv and (y - fv[0]) ** 2 + (x - fv[1]) ** 2 <= (1.5 * fv[2]) ** 2 else "other")
        out.append(dict(name=n, det=det, cls=cls, area=int(st[j, cv2.CC_STAT_AREA]), y=y, x=x))
    return out
FPC = pd.DataFrame([c for det in ("ex", "he") for (g, role), names in ROLE.items() if role == "test" for n in names for c in fp_classes(n, det, mask_for(n, det, BASE))])
if len(FPC):
    FPC["group"] = FPC.name.map(man.set_index("name").group)
    t = FPC.groupby(["det", "group", "cls"]).agg(blobs=("area", "size"), px=("area", "sum")).reset_index()
    t["px_share_pct"] = 100 * t.px / t.groupby(["det", "group"]).px.transform("sum"); display(t.round(1)); t.to_csv(OUT / "stage1_fp_classes.csv", index=False)
LF = []
for det in ("ex", "he"):
    for g in ("IDRiD", "DDR"):
        df, fl = EV[(det, g, "test")]
        for n, f in zip(df.name, fl):
            if not len(f): continue
            _, lab, st, _ = cv2.connectedComponentsWithStats(GT512[(n, det)].astype(np.uint8), connectivity=4)
            for a, ok in zip(st[1:, cv2.CC_STAT_AREA], f):
                LF.append(dict(det=det, group=g, name=n, area512=int(a), found=bool(ok), lesions_in_image=len(f), fov_w=INFO.loc[n, "fov_w"], brightness=INFO.loc[n, "brightness"]))
LF = pd.DataFrame(LF); LF.to_csv(OUT / "stage1_lesion_level.csv", index=False)
LF["size_bin"] = pd.cut(LF.area512, [0, 4, 20, 100, np.inf], labels=["1-4 px", "5-20 px", "21-100 px", ">100 px"])
bt = LF.groupby(["det", "group", "size_bin"], observed=True).found.agg(["sum", "size"]).reset_index()
bt["found_pct"] = 100 * bt["sum"] / bt["size"]; bt[["ci_low", "ci_high"]] = [wilson(a, b) for a, b in zip(bt["sum"], bt["size"])]
display(bt.round(1)); bt.to_csv(OUT / "stage1_recall_by_size.csv", index=False)
pim = LF.groupby(["det", "group", "name"]).agg(recall=("found", "mean"), fov_w=("fov_w", "first"), brightness=("brightness", "first"), lesions=("found", "size")).reset_index()
corr = []
for (det, g), d in pim.groupby(["det", "group"]):
    for v in ("fov_w", "brightness", "lesions"):
        r = stats.spearmanr(d.recall, d[v]); corr.append(dict(det=det, group=g, variable=v, n=len(d), spearman_rho=r.statistic, p=r.pvalue))
CORR = pd.DataFrame(corr); display(CORR.round(3)); CORR.to_csv(OUT / "stage1_recall_correlates.csv", index=False)

## 9. Stage 2: fixes without retraining (validation data only)

In [ ]:
import itertools
VAL = {"DDR": ROLE[("DDR", "val")], "healthy": ROLE[("healthy", "val")]}
grid_rows = []
for det in ("ex", "he"):
    extra = [(False, False)] if det == "ex" else [(False, False), (True, False), (False, True), (True, True)]
    for v, t, rim, ms, (dsc, fv) in itertools.product(CANDIDATES + ["up512"], THR_GRID, (0, 8), (1, 4), extra):
        cfg = dict(variant=v, thr=t, rim=rim, min_size=ms, disc=dsc, fov=fv)
        dd, _ = evaluate(cfg, VAL["DDR"], det); hh, _ = evaluate(cfg, VAL["healthy"], det); d = dd[dd.lesions > 0]
        grid_rows.append(dict(det=det, **cfg, recall_pct=100 * d.found.sum() / d.lesions.sum(), prec_tol_pct=100 * dd.tp_tol.sum() / max(dd.pred_px.sum(), 1),
                              healthy_fp_px=hh.fp_px.mean(), healthy_any_pct=100 * (hh.pred_px > 0).mean()))
GR = pd.DataFrame(grid_rows); GR.to_csv(OUT / "stage2_grid_validation.csv", index=False)
CHOSEN = {}
for det in ("ex", "he"):
    g = GR[GR.det == det]; b = g[(g.variant == "s42") & (g.thr == BASE_THR) & (g.rim == 0) & (g.min_size == 1) & ~g.disc & ~g.fov].iloc[0]
    q = g[(g.variant != "up512") & (g.healthy_fp_px <= b.healthy_fp_px) & (g.prec_tol_pct >= b.prec_tol_pct)]
    best = q.sort_values(["recall_pct", "healthy_fp_px"], ascending=[False, True]).iloc[0] if len(q) else None
    CHOSEN[det] = None if best is None or (best[["variant", "thr", "rim", "min_size", "disc", "fov"]] == b[["variant", "thr", "rim", "min_size", "disc", "fov"]]).all() else \
        dict(variant=best.variant, thr=float(best.thr), rim=int(best.rim), min_size=int(best.min_size), disc=bool(best.disc), fov=bool(best.fov))
    print(det.upper(), "baseline (val):", b[["recall_pct", "prec_tol_pct", "healthy_fp_px", "healthy_any_pct"]].round(2).to_dict())
    print(det.upper(), "chosen:", CHOSEN[det], "" if best is None else best[["recall_pct", "prec_tol_pct", "healthy_fp_px", "healthy_any_pct"]].round(2).to_dict())
    pv = g[(g.rim == 0) & (g.min_size == 1) & ~g.disc & ~g.fov & (g.thr == BASE_THR)][["variant", "recall_pct", "prec_tol_pct", "healthy_fp_px", "healthy_any_pct"]]
    print(det.upper(), "each variant at the production threshold, no post-processing (val):"); display(pv.round(2))
fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
for i, det in enumerate(("ex", "he")):
    g = GR[GR.det == det]
    for v in VARIANTS:
        s = g[(g.variant == v) & (g.rim == 0) & (g.min_size == 1) & ~g.disc & ~g.fov].sort_values("thr")
        ax[i].plot(s.healthy_fp_px, s.recall_pct, marker="o", ms=4, lw=1.2, label=v, ls="--" if v == "up512" else "-")
    b = g[(g.variant == "s42") & (g.thr == BASE_THR) & (g.rim == 0) & (g.min_size == 1) & ~g.disc & ~g.fov].iloc[0]
    ax[i].scatter([b.healthy_fp_px], [b.recall_pct], s=120, facecolors="none", edgecolors="k", lw=2, zorder=5, label="production")
    if CHOSEN[det]:
        c = g[(g.variant == CHOSEN[det]["variant"]) & (g.thr == CHOSEN[det]["thr"]) & (g.rim == CHOSEN[det]["rim"]) & (g.min_size == CHOSEN[det]["min_size"]) & (g.disc == CHOSEN[det]["disc"]) & (g.fov == CHOSEN[det]["fov"])].iloc[0]
        ax[i].scatter([c.healthy_fp_px], [c.recall_pct], marker="*", s=260, c="k", zorder=6, label="chosen")
    ax[i].set(xscale="symlog", xlabel="healthy eyes: false-positive px per eye at 512 (validation)", ylabel="DDR-val lesion recall %", title=f"{det.upper()}: each line = thresholds 0.2-0.8")
    ax[i].legend(frameon=False, fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "stage2_tradeoff.png", dpi=110); plt.show()

## 10. Stage 2 confirmation on the test data (once)

In [ ]:
CONF, CT = [], []
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rank, i in enumerate(o): run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj
for det in ("ex", "he"):
    if CHOSEN[det] is None: print(det.upper(), ": no configuration beat production on validation; nothing to confirm"); continue
    for (g, role), names in ROLE.items():
        if role != "test": continue
        b_df, b_f = evaluate(BASE, names, det, paths=True); c_df, c_f = evaluate(CHOSEN[det], names, det, paths=True)
        CONF.append(dict(det=det, group=g, config="production", **summarise(b_df, g))); CONF.append(dict(det=det, group=g, config="chosen", **summarise(c_df, g)))
        if g == "healthy":
            p_fp = stats.wilcoxon(c_df.fp_px, b_df.fp_px).pvalue if (c_df.fp_px != b_df.fp_px).any() else 1.0
            ab, ac = b_df.pred_px > 0, c_df.pred_px > 0; n01, n10 = int((~ab & ac).sum()), int((ab & ~ac).sum())
            p_any = stats.binomtest(min(n01, n10), n01 + n10).pvalue if n01 + n10 else 1.0
            CT += [dict(det=det, group=g, test="healthy FP px per eye (Wilcoxon)", median_change=float((c_df.fp_px - b_df.fp_px).median()), p=p_fp),
                   dict(det=det, group=g, test="healthy any-detection (McNemar)", median_change=float(n01 - n10), p=p_any)]
        else:
            fb, fc = np.concatenate([f for f in b_f if len(f)]), np.concatenate([f for f in c_f if len(f)])
            n01, n10 = int((~fb & fc).sum()), int((fb & ~fc).sum())
            CT += [dict(det=det, group=g, test="lesions found (McNemar)", median_change=float(n01 - n10), p=stats.binomtest(min(n01, n10), n01 + n10).pvalue if n01 + n10 else 1.0),
                   dict(det=det, group=g, test="FP px per image (Wilcoxon)", median_change=float((c_df.fp_px - b_df.fp_px).median()),
                        p=stats.wilcoxon(c_df.fp_px, b_df.fp_px).pvalue if (c_df.fp_px != b_df.fp_px).any() else 1.0)]
CONF, CT = pd.DataFrame(CONF), pd.DataFrame(CT)
if len(CT): CT["p_holm"] = holm(CT.p); CT["significant"] = CT.p_holm < ALPHA
display(CONF.round(2)); display(CT.round(4)); CONF.to_csv(OUT / "stage2_test_confirmation.csv", index=False); CT.to_csv(OUT / "stage2_tests.csv", index=False)

## 11. Cases for Victoria (production detectors, test data)
Picked by rule: per detector, the 6 largest false-positive blobs on healthy eyes (at most one per eye), the 4 largest missed GT lesions and the 4 largest false-positive blobs on diseased test images. Each case shows the 512 crop and the native-resolution crop with the detected blob (or missed grader lesion) outlined 2 px outside its edge, plus a clean native crop. Form `notebooks/inputs/detector_case_review_form.csv` (created once): `verdict` = for false positives 1 real lesion / 0 not a lesion / 9 unsure; for misses 1 lesion clearly visible / 0 not visible or doubtful / 9 unsure.

In [ ]:
CASE_KEY, CASE_FORM, CASES = INPUTS / "detector_case_review_key.csv", INPUTS / "detector_case_review_form.csv", OUT / "case_sheets"
if not SUBSET:
    if CASE_KEY.exists(): ck = pd.read_csv(CASE_KEY)
    else:
        picks = []
        for det in ("ex", "he"):
            if not len(FPC): continue
            f = FPC[(FPC.det == det) & (FPC.group == "healthy")].sort_values("area", ascending=False).drop_duplicates("name").head(6); picks.append(f.assign(kind="false positive, healthy"))
            f = FPC[(FPC.det == det) & (FPC.group != "healthy")].sort_values("area", ascending=False).drop_duplicates("name").head(4); picks.append(f.assign(kind="false positive, diseased"))
            ms = []
            for g in ("IDRiD", "DDR"):
                df, fl = EV[(det, g, "test")]
                for n, fv_ in zip(df.name, fl):
                    if not len(fv_): continue
                    _, lab, st, cen = cv2.connectedComponentsWithStats(GT512[(n, det)].astype(np.uint8), connectivity=4)
                    ms += [dict(name=n, det=det, area=int(st[j + 1, cv2.CC_STAT_AREA]), y=int(round(cen[j + 1][1])), x=int(round(cen[j + 1][0])), group=g) for j in np.nonzero(~fv_)[0]]
            if ms: picks.append(pd.DataFrame(ms).sort_values("area", ascending=False).drop_duplicates("name").head(4).assign(kind="missed lesion"))
        ck = pd.concat(picks, ignore_index=True)[["name", "group", "det", "kind", "area", "y", "x"]]; ck.insert(0, "case", np.arange(1, len(ck) + 1)); ck.to_csv(CASE_KEY, index=False)
    if not CASE_FORM.exists(): pd.DataFrame({"case": ck.case, "verdict": "", "notes": ""}).to_csv(CASE_FORM, index=False)
    CASES.mkdir(parents=True, exist_ok=True)
    def case_component(c):                           # the exact detected blob (false positives) or grader lesion (misses) of this case
        m = mask_for(c.name, c.det, BASE) if c.kind.startswith("false positive") else GT512[(c.name, c.det)]
        k, lab, st, cen = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=4)
        j = 1 + int(np.argmin((cen[1:, 0] - c.x) ** 2 + (cen[1:, 1] - c.y) ** 2)); return lab == j
    def outline(a, m, color="#00e5ff", lw=1.2):
        for cnt in cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[0]:
            cnt = cnt[:, 0, :]
            if len(cnt) > 2: a.plot(np.r_[cnt[:, 0], cnt[0, 0]], np.r_[cnt[:, 1], cnt[0, 1]], color=color, lw=lw)
    K2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    def case_sheet(chunk, path):
        fig, ax = plt.subplots(len(chunk), 3, figsize=(20, 6.6 * len(chunk))); ax = np.atleast_2d(ax)
        for i, c in enumerate(chunk.itertuples()):
            cm = case_component(c); ys, xs = np.nonzero(cm)
            cy, cx = (ys.min() + ys.max()) // 2, (xs.min() + xs.max()) // 2
            R5 = int(min(128, max(24, np.ceil(max(np.ptp(ys), np.ptp(xs)) / 2) + 12)))      # crop sized to the blob + margin
            y0, x0 = int(np.clip(cy - R5, 0, 512 - 2 * R5)), int(np.clip(cx - R5, 0, 512 - 2 * R5))
            ring = (cv2.dilate(cm.astype(np.uint8), K2) > 0)[y0:y0 + 2 * R5, x0:x0 + 2 * R5]  # drawn 2 px outside, never over the blob
            H, W = int(INFO.loc[c.name, "H"]), int(INFO.loc[c.name, "W"]); fy, fx = H / 512, W / 512
            ny0, nx0, ny1, nx1 = int(y0 * fy), int(x0 * fx), int(np.ceil((y0 + 2 * R5) * fy)), int(np.ceil((x0 + 2 * R5) * fx))
            nat = cv2.cvtColor(cv2.imread(str(ROOT / man.set_index("name").loc[c.name, "image"])), cv2.COLOR_BGR2RGB)[ny0:ny1, nx0:nx1]
            ring_n = cv2.resize(ring.astype(np.uint8), (nat.shape[1], nat.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
            what = "detection" if c.kind.startswith("false positive") else "grader lesion"
            ax[i, 0].imshow(RGB512[c.name][y0:y0 + 2 * R5, x0:x0 + 2 * R5], interpolation="nearest"); outline(ax[i, 0], ring)
            ax[i, 1].imshow(nat); outline(ax[i, 1], ring_n); ax[i, 2].imshow(nat)
            ax[i, 0].set_title(f"Case {c.case}: {c.det.upper()} {c.kind}
512 control, {what} outlined", fontsize=12)
            ax[i, 1].set_title(f"native resolution, {what} outlined", fontsize=12); ax[i, 2].set_title("native resolution, clean", fontsize=12)
            for a in ax[i]: a.set_xticks([]); a.set_yticks([])
        fig.tight_layout(); fig.savefig(path, dpi=100); plt.close(fig)
    for s0 in range(0, len(ck), 4): case_sheet(ck.iloc[s0:s0 + 4], CASES / f"cases_{s0 // 4 + 1:02d}.png")
    print(len(ck), "cases on", int(np.ceil(len(ck) / 4)), "sheets in", CASES)
    cf = pd.read_csv(CASE_FORM, dtype=str).fillna(""); cdone = cf[cf.verdict.str.strip() != ""].astype({"case": int})
    CASE_RES = None if cdone.empty else cdone.merge(ck, on="case").assign(verdict=lambda d: d.verdict.astype(int))
    if CASE_RES is None: print("Case review pending.")
    else: display(CASE_RES.groupby(["det", "kind"]).verdict.value_counts().unstack(fill_value=0))

## 12. Claims (the only source for messages)

In [ ]:
claims = []
def add(kind, text, evidence): claims.append(dict(kind=kind, claim=text, evidence=evidence))
if SUBSET:
    print("SUBSET MODE: no claims")
else:
    add("measured", f"Record reproduced: HE IDRiD-test recall {rhu(R['he_idrid']['recall'], 4)}, HE DDR-test {R['he_ddr']['hits']}/{R['he_ddr']['blobs']} blobs, EX IDRiD-test recall {rhu(R['ex_idrid_raw']['recall'], 4)}.", "Section 5 asserts")
    add("measured", f"First DDR measurement of EX (full resolution, disc-excluded, {R['ex_ddr_disc']['images']} DDR test images with exudates): blob recall {rhu(100 * R['ex_ddr_disc']['recall'])}% ({R['ex_ddr_disc']['hits']}/{R['ex_ddr_disc']['blobs']}), pooled precision {rhu(100 * R['ex_ddr_disc']['prec_pooled'])}%.", "Section 5")
    for _, r in S1.iterrows():
        if r.group == "healthy":
            add("measured", f"{r.det.upper()} on healthy eyes ({r.role}, {r.images} eyes): fires on {rhu(r.eyes_with_any_pct)}% of eyes (95% CI {rhu(r.any_ci_low)}-{rhu(r.any_ci_high)}), {rhu(r.fp_px_per_eye, 1)} false-positive px per eye at 512.", "Section 8")
        else:
            add("measured", f"{r.det.upper()} on {r.group} {r.role} ({r.images} images, {r.lesions} lesions): finds {rhu(r.recall_pct)}% of lesions (95% CI {rhu(r.recall_ci_low)}-{rhu(r.recall_ci_high)}); {rhu(r.prec_tol_pct)}% of detected px are on or within 2 px of a grader lesion.", "Section 8, production configuration at 512")
    for det in ("ex", "he"):
        add("measured", f"{det.upper()} Stage 2: " + ("no configuration beat production on validation (healthy false positives and precision no worse, recall higher)." if CHOSEN[det] is None else f"chosen on validation: {CHOSEN[det]}."), "Section 9 selection rule")
    for _, r in CT.iterrows() if len(CT) else []:
        add("measured", f"{r.det.upper()} {r.group} test, chosen vs production, {r.test}: change {rhu(r.median_change, 1)}, Holm p = {r.p_holm:.3g} ({'difference' if r.significant else 'no detectable difference'}).", "Section 10")
    if CASE_RES is not None:
        for (det, kind), d in CASE_RES.groupby(["det", "kind"]):
            add("clinical (Victoria)", f"{det.upper()} {kind}: verdicts " + ", ".join(f"{int((d.verdict == v).sum())}x{v}" for v in (1, 0, 9)) + f" of {len(d)} cases.", "Section 11 case review")
    else:
        add("untested", "Whether the healthy-eye false positives shown are real lesions (EyePACS grade 0 labels are imperfect).", "Pending Victoria's case review")
    claims = pd.DataFrame(claims); claims.to_csv(OUT / "claims.csv", index=False)
    with pd.option_context("display.max_colwidth", None): display(claims)